# Navegación sobre terreno real con Deep Q-Network (DQN)
**Curso MIA-204 — Aprendizaje por Refuerzo · Trabajo final**

Un agente aprende a navegar entre **cualquier par de puntos (A, B)** de un terreno
real (faldas del volcán Misti, Arequipa — DEM Copernicus GLO-30, 30 m), minimizando
el esfuerzo por pendiente. Es la continuación del trabajo parcial (Q-learning tabular
con meta fija): aquí la meta cambia en cada episodio, el agente solo **ve un parche
local de 9×9** más un vector hacia la meta, y la tabla Q se reemplaza por una
**red neuronal** entrenada con DQN (Mnih et al., 2015).

Componentes (terminología de la clase 5.1):
- **Reproducción de Experiencias** (replay buffer)
- **Objetivos-Q Fijados** (red objetivo θ⁻)
- **Double DQN** (opcional, para las ablaciones)

> ▶ Corre en **Google Colab** (con o sin GPU) o localmente. Todo se descarga y
> configura solo. Código idéntico a los módulos `dqn/*.py` del repositorio.


In [ ]:
 %pip install -q rasterio

In [ ]:
# ── Setup: dependencias, imports, dispositivo ──────────────────────────
# En Colab solo falta rasterio; local ya suele estar todo.
import rasterio
import heapq, os, random, time
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# GPU de Colab (cuda) si hay; si no, CPU. Ojo: NO usamos MPS (GPU de Mac)
# a propósito — la red es tan chica (~27k pesos) que el costo de mover
# datos a la GPU supera al cálculo: medimos 2.3 s/episodio en MPS contra
# 0.4 s/episodio en CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | dispositivo: {device}")

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); random.seed(SEMILLA)


## 1 · El terreno real

Recorte de ~3×3 km del DEM **Copernicus GLO-30** en las faldas del Misti
(108×108 celdas de 30 m). Se lee directo desde AWS Open Data (público, sin
login) usando ventanas COG — solo se descargan los ~50 KB de la zona.


In [ ]:
# ── Descargar el DEM si no está ────────────────────────────────────────
URL = ("https://copernicus-dem-30m.s3.amazonaws.com/"
       "Copernicus_DSM_COG_10_S17_00_W072_00_DEM/"
       "Copernicus_DSM_COG_10_S17_00_W072_00_DEM.tif")
BBOX = (-71.475, -16.375, -71.445, -16.345)   # lon/lat: faldas del Misti

RUTA_DEM = next((r for r in ["data/dem.tif", "../data/dem.tif", "dem.tif"]
                 if os.path.exists(r)), None)
if RUTA_DEM is None:
    from rasterio.windows import from_bounds
    print("Descargando recorte del DEM desde AWS Open Data...")
    with rasterio.open(URL) as src:
        ventana = from_bounds(*BBOX, transform=src.transform)
        elev = src.read(1, window=ventana)
        perfil = src.profile.copy()
        perfil.update(height=elev.shape[0], width=elev.shape[1],
                      transform=src.window_transform(ventana))
    with rasterio.open("dem.tif", "w", **perfil) as dst:
        dst.write(elev, 1)
    RUTA_DEM = "dem.tif"
print(f"DEM: {RUTA_DEM}")


In [ ]:
# ── Mirar el terreno ───────────────────────────────────────────────────
with rasterio.open(RUTA_DEM) as src:
    elevacion = src.read(1).astype(float)
gy, gx = np.gradient(elevacion, 30.0)
pendiente = np.sqrt(gx**2 + gy**2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))
im1 = a1.imshow(elevacion, cmap="terrain"); a1.set_title("Elevación (m)")
plt.colorbar(im1, ax=a1, shrink=0.8)
im2 = a2.imshow(pendiente, cmap="magma"); a2.set_title("Pendiente (m/m)")
plt.colorbar(im2, ax=a2, shrink=0.8)
plt.tight_layout(); plt.show()
print(f"{elevacion.shape[0]}x{elevacion.shape[1]} celdas | "
      f"elevación {elevacion.min():.0f}–{elevacion.max():.0f} m")


## 2 · El entorno (MDP)

Cambios respecto al parcial — empujan hacia la red:

| | Parcial (tabular) | Final (DQN) |
|---|---|---|
| Resolución | 54×54 (promediado) | **108×108** completa |
| Inicio | fijo (A) | **aleatorio por episodio** |
| Meta | fija (B) | fija (B) — misma para todos los episodios |
| Estado | (fila, col) | **parche 9×9 + vector a la meta = 83 números** |

Elegimos **meta fija + inicio aleatorio**. Con una sola meta, la función de valor
es un único campo suave (como la V tabular que sí convergía) y DQN entrena **estable**.
El inicio aleatorio hace el problema no trivial: el agente aprende a llegar desde
**cualquier** celda viendo solo su parche local, así la red **generaliza** — una tabla
necesitaría memorizar cada celda de salida, la red aprende "leer terreno y caminar
hacia la meta". (El modo meta 100% aleatoria existe con `meta_fija=False`; lo dejamos
como experimento-extra porque DQN diverge ahí — ver nota de shaping abajo.)

Recompensas (escala /10 del parcial, mejor para la red): paso ≈ −0.1…−0.6
según pendiente, choque −5, meta +10. `prob_resbalon > 0` activa la
transición estocástica (resbala más donde está más empinado).

**Reward shaping** (la lección más valiosa del proyecto): sin un bono denso por
acercarse, el agente encuentra B por azar muy rara vez y no aprende. Probamos primero
el shaping **potencial** de Ng et al. (1999), Φ(s) = −k·dist — el "de libro". **Falló:**
con γ<1 deja un residuo positivo (1−γ)·k·dist por estar lejos, así que hacer *ping-pong*
lejos de la meta **gana** recompensa; el agente aprendía a oscilar y su política greedy
se trababa en 2-ciclos. Lo cambiamos por shaping de **progreso**, +k·(dist_antes −
dist_después): telescopea a 0 en cualquier ciclo, así el ping-pong queda penalizado por
el costo del paso. Un **curriculum** (inicios cerca primero, se alejan) acelera el
aprendizaje. Con meta fija + progreso el examen llega a **100% (20/20)** y se mantiene.


In [ ]:
ACCIONES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # ↑ ↓ ← →
NUM_ACCIONES = len(ACCIONES)


class TerrenoDRLEnv:
    """Gridworld sobre DEM real con metas aleatorias y observación local."""

    def __init__(self, ruta_dem, tam_celda=30.0, peso_pendiente=8.0,
                 umbral_obstaculo=0.60, parche=9, max_pasos=500,
                 dist_minima=15, prob_resbalon=0.0, k_shaping=0.2,
                 gamma=0.99, meta_fija=True, semilla=0):
        with rasterio.open(ruta_dem) as src:
            self.elevacion = src.read(1).astype(float)
        self.filas, self.columnas = self.elevacion.shape

        gy, gx = np.gradient(self.elevacion, tam_celda)
        self.pendiente = np.sqrt(gx**2 + gy**2)
        self.costo = 1.0 + peso_pendiente * self.pendiente
        self.obstaculo = self.pendiente > umbral_obstaculo

        # lo que el agente VE: pendiente 0-1; obstáculo y borde = 1.0
        self.vista = np.clip(self.pendiente / umbral_obstaculo, 0.0, 1.0)
        self.vista[self.obstaculo] = 1.0

        self.parche, self.radio = parche, parche // 2
        self.dim_obs = parche * parche + 2
        self.max_pasos = max_pasos
        self.dist_minima = dist_minima
        self.prob_resbalon = prob_resbalon
        self.k_shaping = k_shaping
        self.gamma = gamma
        self.rng = np.random.default_rng(semilla)

        self.libres = self._componente_mas_grande()

        # Meta FIJA (modo estable): una sola celda destino para todos los
        # episodios -> la funcion de valor es un unico campo suave y DQN
        # entrena estable (sin la divergencia que medimos con metas
        # aleatorias). El inicio SI es aleatorio: el agente aprende a llegar
        # desde cualquier celda, asi la red generaliza.
        self.meta_fija = None
        if meta_fija:
            objetivo = np.array([self.filas - 4, self.columnas - 4])
            d = np.abs(self.libres - objetivo).sum(axis=1)
            self.meta_fija = tuple(self.libres[d.argmin()])

        self.estado = self.meta = None
        self.pasos = 0

    def _componente_mas_grande(self):
        # A y B siempre se sortean de la componente conexa mayor de celdas
        # libres: así el camino entre ellos existe garantizado.
        visitado = np.zeros_like(self.obstaculo)
        mejor = []
        for celda in map(tuple, np.argwhere(~self.obstaculo)):
            if visitado[celda]:
                continue
            frontera, comp = [celda], [celda]
            visitado[celda] = True
            while frontera:
                f, c = frontera.pop()
                for df, dc in ACCIONES.values():
                    v = (f + df, c + dc)
                    if (0 <= v[0] < self.filas and 0 <= v[1] < self.columnas
                            and not self.obstaculo[v] and not visitado[v]):
                        visitado[v] = True
                        frontera.append(v); comp.append(v)
            if len(comp) > len(mejor):
                mejor = comp
        return np.array(mejor)

    def reset(self, inicio=None, meta=None, dist_maxima=None):
        # dist_maxima: tope de distancia inicio-meta, usado por el CURRICULUM
        # (inicios cercanos primero, se alejan con el entrenamiento).
        if meta is None:
            meta = self.meta_fija
        if inicio is None:
            while True:
                inicio = tuple(self.libres[self.rng.integers(len(self.libres))])
                d = abs(inicio[0]-meta[0]) + abs(inicio[1]-meta[1])
                if d >= self.dist_minima and (dist_maxima is None or d <= dist_maxima):
                    break
        self.estado, self.meta, self.pasos = tuple(inicio), tuple(meta), 0
        return self._observar()

    def _observar(self):
        r = self.radio
        fila, col = self.estado
        vista = np.pad(self.vista, r, constant_values=1.0)  # fuera del mapa = pared
        patch = vista[fila:fila + 2*r + 1, col:col + 2*r + 1]
        delta = np.array([(self.meta[0]-fila)/self.filas,
                          (self.meta[1]-col)/self.columnas])
        return np.concatenate([patch.ravel(), delta]).astype(np.float32)

    def _dist_meta(self, celda):
        return abs(celda[0] - self.meta[0]) + abs(celda[1] - self.meta[1])

    def step(self, accion):
        if self.prob_resbalon > 0:
            if self.rng.random() < self.prob_resbalon * self.vista[self.estado]:
                accion = int(self.rng.integers(NUM_ACCIONES))

        df, dc = ACCIONES[accion]
        fila, col = self.estado
        nueva = (fila + df, col + dc)
        fuera = not (0 <= nueva[0] < self.filas and 0 <= nueva[1] < self.columnas)
        if fuera or self.obstaculo[nueva]:
            recompensa, nueva = -5.0, self.estado
        elif nueva == self.meta:
            recompensa = 10.0
        else:
            recompensa = -self.costo[nueva] / 10.0

        # Reward shaping de PROGRESO: +k por acercarse, -k por alejarse.
        # (Sin esto el agente casi nunca encuentra B por azar en 108x108 y no
        # hay premio del cual aprender.) Probamos primero el shaping potencial
        # de Ng et al. 1999 y falló: con gamma<1 dejaba un residuo positivo
        # por estar lejos, y hacer ping-pong lejos GANABA recompensa -> la
        # politica greedy se trababa en 2-ciclos. El shaping de progreso
        # telescopea a 0 en cualquier ciclo, sin esa trampa.
        if self.k_shaping > 0:
            recompensa += self.k_shaping * (
                self._dist_meta(self.estado) - self._dist_meta(nueva)
            )

        self.estado = nueva
        self.pasos += 1
        llego = nueva == self.meta
        return self._observar(), recompensa, llego or self.pasos >= self.max_pasos, llego


In [ ]:
# ── Qué ve el agente: un ejemplo ───────────────────────────────────────
env = TerrenoDRLEnv(RUTA_DEM, semilla=SEMILLA)
obs = env.reset()
print(f"Mapa {env.filas}x{env.columnas} | libres conectadas: {len(env.libres)} | "
      f"obs: {obs.shape[0]} números")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))
a1.imshow(env.vista, cmap="terrain")
a1.plot(env.estado[1], env.estado[0], "o", color="red", ms=8, label="A (agente)")
a1.plot(env.meta[1], env.meta[0], "*", color="gold", ms=16,
        markeredgecolor="black", label="B (meta)")
r = env.radio
a1.add_patch(plt.Rectangle((env.estado[1]-r-.5, env.estado[0]-r-.5),
                           env.parche, env.parche, fill=False, ec="red", lw=2))
a1.legend(); a1.set_title("El mapa (el agente NO lo ve entero)")
a2.imshow(obs[:-2].reshape(env.parche, env.parche), cmap="terrain", vmin=0, vmax=1)
a2.set_title(f"Lo que la red recibe: parche {env.parche}x{env.parche}\n"
             f"+ vector a la meta ({obs[-2]:+.2f}, {obs[-1]:+.2f})")
a2.set_xticks([]); a2.set_yticks([])
plt.tight_layout(); plt.show()


## 3 · La red y el agente DQN

- **RedQ**: 83 → 128 → 128 → 4. Una pasada devuelve los 4 valores Q del estado.
- **BufferReproduccion**: memoria con capacidad fija; se entrena con mini-batches aleatorios.
- **AgenteDQN**: update TD (el mismo del parcial) como pérdida Huber + gradiente,
  red objetivo θ⁻ sincronizada cada 1,000 updates.

Banderas de ablación: `usar_replay=False`, `usar_target=False`, `doble=True`.


In [ ]:
class RedQ(nn.Module):
    def __init__(self, dim_obs, num_acciones, oculto=128):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(dim_obs, oculto), nn.ReLU(),
            nn.Linear(oculto, oculto), nn.ReLU(),
            nn.Linear(oculto, num_acciones),
        )

    def forward(self, x):
        return self.capas(x)


class BufferReproduccion:
    def __init__(self, capacidad):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *transicion):
        self.memoria.append(transicion)

    def muestrear(self, n):
        obs, acc, rec, sig, fin = zip(*random.sample(self.memoria, n))
        return (torch.as_tensor(np.array(obs)),
                torch.as_tensor(acc, dtype=torch.long),
                torch.as_tensor(rec, dtype=torch.float32),
                torch.as_tensor(np.array(sig)),
                torch.as_tensor(fin, dtype=torch.float32))

    def __len__(self):
        return len(self.memoria)


class AgenteDQN:
    def __init__(self, dim_obs, num_acciones, gamma=0.99, lr=1e-3, batch=64,
                 capacidad_buffer=100_000, pasos_calentamiento=2_000,
                 sincronizar_cada=1_000, usar_replay=True, usar_target=True,
                 doble=False, device="cpu", semilla=0):
        torch.manual_seed(semilla); random.seed(semilla)
        self.rng = np.random.default_rng(semilla)
        self.device = torch.device(device)
        self.red = RedQ(dim_obs, num_acciones).to(self.device)
        self.red_objetivo = RedQ(dim_obs, num_acciones).to(self.device)
        self.red_objetivo.load_state_dict(self.red.state_dict())
        self.red_objetivo.eval()
        self.optimizador = torch.optim.Adam(self.red.parameters(), lr=lr)
        self.buffer = BufferReproduccion(capacidad_buffer)
        self.num_acciones, self.gamma, self.batch = num_acciones, gamma, batch
        self.pasos_calentamiento = pasos_calentamiento
        self.sincronizar_cada = sincronizar_cada
        self.usar_replay, self.usar_target, self.doble = usar_replay, usar_target, doble
        self.updates = 0

    def elegir_accion(self, obs, epsilon):
        if self.rng.random() < epsilon:
            return int(self.rng.integers(self.num_acciones))
        with torch.no_grad():
            x = torch.as_tensor(obs, device=self.device).unsqueeze(0)
            return int(self.red(x).argmax().item())

    def observar_y_entrenar(self, obs, accion, recompensa, obs_sig, terminado):
        self.buffer.guardar(obs, accion, recompensa, obs_sig, terminado)
        if len(self.buffer) < max(self.batch, self.pasos_calentamiento):
            return None
        if self.usar_replay:
            lote = self.buffer.muestrear(self.batch)
        else:   # ablación: solo la última transición (correlacionado)
            lote = (torch.as_tensor(np.array([obs])),
                    torch.as_tensor([accion], dtype=torch.long),
                    torch.as_tensor([recompensa], dtype=torch.float32),
                    torch.as_tensor(np.array([obs_sig])),
                    torch.as_tensor([float(terminado)], dtype=torch.float32))
        return self._update(*(t.to(self.device) for t in lote))

    def _update(self, obs, acc, rec, sig, fin):
        with torch.no_grad():
            red_eval = self.red_objetivo if self.usar_target else self.red
            if self.doble:      # θ elige, θ⁻ evalúa
                mejores = self.red(sig).argmax(dim=1, keepdim=True)
                q_sig = red_eval(sig).gather(1, mejores).squeeze(1)
            else:
                q_sig = red_eval(sig).max(dim=1).values
            objetivo = rec + self.gamma * q_sig * (1.0 - fin)

        prediccion = self.red(obs).gather(1, acc.unsqueeze(1)).squeeze(1)
        perdida = nn.functional.smooth_l1_loss(prediccion, objetivo)
        self.optimizador.zero_grad()
        perdida.backward()
        nn.utils.clip_grad_norm_(self.red.parameters(), 10.0)
        self.optimizador.step()

        self.updates += 1
        if self.usar_target and self.updates % self.sincronizar_cada == 0:
            self.red_objetivo.load_state_dict(self.red.state_dict())   # θ⁻ ← θ
        return float(perdida.item())


## 4 · Entrenamiento

ε baja linealmente de 1.0 a 0.05 durante los primeros 150k **pasos** (no
episodios) y se queda ahí — cada episodio es una meta nueva, siempre conviene
explorar un poco. Cada 100 episodios: **examen** sin exploración sobre 20 pares
(A, B) fijos que el entrenamiento nunca sortea.

Tiempo aprox. de la corrida base (3,000 episodios): ~15–25 min en Colab GPU/CPU.
Para probar rápido, baja `EPISODIOS` a 500.


In [ ]:
def pares_de_evaluacion(env, n=20, semilla=12345):
    """
    Pares (inicio, meta) de examen, con OTRA semilla que el entrenamiento.
    En modo meta fija, la meta es siempre la misma y solo varia el inicio:
    medimos si el agente llega desde inicios nunca vistos.
    """
    rng = np.random.default_rng(semilla)
    pares = []
    while len(pares) < n:
        ini = tuple(env.libres[rng.integers(len(env.libres))])
        meta = env.meta_fija if env.meta_fija is not None else             tuple(env.libres[rng.integers(len(env.libres))])
        if abs(ini[0]-meta[0]) + abs(ini[1]-meta[1]) >= env.dist_minima:
            pares.append((ini, meta))
    return pares


def evaluar(env, agente, pares):
    """Examen greedy (ε=0). Devuelve (tasa de éxito, pasos promedio)."""
    exitos, pasos = 0, 0
    for ini, meta in pares:
        obs, fin = env.reset(inicio=ini, meta=meta), False
        while not fin:
            obs, _, fin, llego = env.step(agente.elegir_accion(obs, 0.0))
        exitos += int(llego); pasos += env.pasos
    return exitos / len(pares), pasos / len(pares)


def entrenar(env, agente, episodios, eps_fin=0.05, eps_pasos=150_000,
             eval_cada=100, verbose=True):
    pares = pares_de_evaluacion(env)
    historia = {"recompensa": [], "pasos": [], "exito": [],
                "eval_ep": [], "eval_exito": [], "eval_pasos": []}
    paso_global, t0 = 0, time.time()
    # Curriculum: la meta arranca cerca (20 celdas) y se aleja hasta cubrir
    # todo el mapa en la primera mitad del entrenamiento.
    curr_ini, curr_fin, curr_hasta = 20, env.filas + env.columnas, episodios // 2

    for ep in range(1, episodios + 1):
        dist_max = curr_ini + min(1.0, ep/curr_hasta) * (curr_fin - curr_ini)
        obs, total, fin = env.reset(dist_maxima=dist_max), 0.0, False
        while not fin:
            eps = max(eps_fin, 1.0 - paso_global / eps_pasos)
            acc = agente.elegir_accion(obs, eps)
            obs_sig, rec, fin, llego = env.step(acc)
            agente.observar_y_entrenar(obs, acc, rec, obs_sig, fin and llego)
            obs, total = obs_sig, total + rec
            paso_global += 1
        historia["recompensa"].append(total)
        historia["pasos"].append(env.pasos)
        historia["exito"].append(int(llego))

        if ep % eval_cada == 0 or ep == episodios:
            ex, pa = evaluar(env, agente, pares)
            historia["eval_ep"].append(ep)
            historia["eval_exito"].append(ex)
            historia["eval_pasos"].append(pa)
            if verbose:
                print(f"ep {ep:5d} | recompensa {total:8.1f} | ε {eps:.2f} | "
                      f"examen {ex:5.0%} éxito, {pa:5.0f} pasos | "
                      f"{time.time()-t0:5.0f} s")
    return historia


In [ ]:
# ── Corrida base: DQN completo ─────────────────────────────────────────
EPISODIOS = 3000        # baja a 500 para una prueba rápida

env = TerrenoDRLEnv(RUTA_DEM, semilla=SEMILLA)
agente = AgenteDQN(env.dim_obs, NUM_ACCIONES, device=device, semilla=SEMILLA)
historia = entrenar(env, agente, EPISODIOS)
torch.save(agente.red.state_dict(), "dqn_base.pt")


In [ ]:
# ── Curvas de aprendizaje ──────────────────────────────────────────────
def suavizar(x, k=100):
    """Media móvil; devuelve (xs, ys) listos para plotear."""
    k = min(k, len(x))
    return range(k - 1, len(x)), np.convolve(x, np.ones(k)/k, mode="valid")

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0,0].plot(historia["recompensa"], alpha=.25, lw=.5)
ax[0,0].plot(*suavizar(historia["recompensa"]))
ax[0,0].set_title("Recompensa por episodio (media móvil 100)")
ax[0,1].plot(*suavizar(historia["exito"]))
ax[0,1].set_ylim(0, 1.05); ax[0,1].set_title("Tasa de éxito en entrenamiento")
ax[1,0].plot(*suavizar(historia["pasos"]))
ax[1,0].set_title("Pasos por episodio")
ax[1,1].plot(historia["eval_ep"], historia["eval_exito"], "o-")
ax[1,1].set_ylim(0, 1.05)
ax[1,1].set_title("EXAMEN: éxito en 20 pares (A,B) nunca entrenados")
for a in ax.ravel():
    a.set_xlabel("episodio"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 5 · Evaluación: rutas aprendidas y comparación con el óptimo

Dibujamos rutas greedy en pares nunca vistos y las comparamos contra el
**óptimo exacto de Dijkstra** (que usa el mapa completo — el agente solo vio
parches). La brecha de costo mide qué tan cerca del óptimo navega la red.


In [ ]:
def ruta_greedy(env, agente, inicio, meta):
    obs, fin = env.reset(inicio=inicio, meta=meta), False
    camino = [env.estado]
    while not fin:
        obs, _, fin, llego = env.step(agente.elegir_accion(obs, 0.0))
        camino.append(env.estado)
    return camino, llego


def dijkstra(env, inicio, meta):
    """Óptimo exacto con el mapa completo (la vara de medir)."""
    dist = {inicio: 0.0}
    previo = {}
    cola = [(0.0, inicio)]
    while cola:
        d, u = heapq.heappop(cola)
        if u == meta:
            break
        if d > dist.get(u, float("inf")):
            continue
        for df, dc in ACCIONES.values():
            v = (u[0]+df, u[1]+dc)
            if not (0 <= v[0] < env.filas and 0 <= v[1] < env.columnas):
                continue
            if env.obstaculo[v]:
                continue
            nd = d + env.costo[v]
            if nd < dist.get(v, float("inf")):
                dist[v], previo[v] = nd, u
                heapq.heappush(cola, (nd, v))
    camino, c = [meta], meta
    while c != inicio:
        c = previo[c]
        camino.append(c)
    return camino[::-1], dist[meta]


def costo_camino(env, camino):
    return float(sum(env.costo[c] for c in camino[1:]))


In [ ]:
# ── 4 rutas de examen sobre el terreno ─────────────────────────────────
pares = pares_de_evaluacion(env)[:4]
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for axi, (ini, meta) in zip(axes.ravel(), pares):
    camino, llego = ruta_greedy(env, agente, ini, meta)
    cam_dij, costo_opt = dijkstra(env, ini, meta)
    axi.imshow(env.vista, cmap="terrain")
    axi.plot([c[1] for c in cam_dij], [c[0] for c in cam_dij],
             "--", color="white", lw=1.5, label=f"Dijkstra ({costo_opt:.0f})")
    etiqueta = f"DQN ({costo_camino(env, camino):.0f})" if llego else "DQN (no llegó)"
    axi.plot([c[1] for c in camino], [c[0] for c in camino],
             color="red", lw=2, label=etiqueta)
    axi.plot(ini[1], ini[0], "o", color="red", ms=7)
    axi.plot(meta[1], meta[0], "*", color="gold", ms=15, markeredgecolor="black")
    brecha = (costo_camino(env, camino) / costo_opt - 1) * 100 if llego else float("nan")
    axi.set_title(f"A={ini} → B={meta}" + (f" | brecha {brecha:+.0f}%" if llego else ""))
    axi.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()


## 6 · Ablaciones (el experimento de Liu & Zou, en nuestro terreno)

¿Cuánto aporta cada mecanismo? Entrenamos 4 configuraciones — DQN completo,
sin replay, sin red objetivo, y Double DQN — con menos episodios (para que sea
viable en clase) y comparamos el examen final.

⚠ ~4× el tiempo de una corrida. Súbele `EPISODIOS_AB` para el informe final.


In [ ]:
EPISODIOS_AB = 1000       # para el informe: 3000
CONFIGS = {
    "DQN completo":  dict(),
    "sin replay":    dict(usar_replay=False),
    "sin θ⁻":        dict(usar_target=False),
    "Double DQN":    dict(doble=True),
}
resultados = {}
for nombre, extra in CONFIGS.items():
    print(f"\n════ {nombre} ════")
    env_a = TerrenoDRLEnv(RUTA_DEM, semilla=SEMILLA)
    ag = AgenteDQN(env_a.dim_obs, NUM_ACCIONES, device=device,
                   semilla=SEMILLA, **extra)
    resultados[nombre] = entrenar(env_a, ag, EPISODIOS_AB, eval_cada=200)


In [ ]:
# ── Comparación de las 4 configuraciones ───────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for nombre, h in resultados.items():
    a1.plot(*suavizar(h["recompensa"]), label=nombre)
    a2.plot(h["eval_ep"], h["eval_exito"], "o-", label=nombre)
a1.set_title("Recompensa (media móvil 100)"); a1.set_xlabel("episodio")
a2.set_title("Examen: éxito en pares nunca vistos"); a2.set_xlabel("episodio")
a2.set_ylim(0, 1.05)
for a in (a1, a2):
    a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

print("Examen final:")
for nombre, h in resultados.items():
    print(f"  {nombre:15s} {h['eval_exito'][-1]:6.0%} éxito | "
          f"{h['eval_pasos'][-1]:5.0f} pasos promedio")


## 6.5 · Experimento extra: transición estocástica (resbalón)

Hasta aquí el mundo era **determinista**: la acción siempre sale como se pidió. Subimos
la complejidad haciéndolo **estocástico**: con probabilidad proporcional a la pendiente,
el paso *resbala* y sale hacia un lado al azar (`prob_resbalon`). Es donde el RL
model-free brilla — no hay modelo del mundo, el agente debe aprender una política
**robusta al azar**. Comparamos DQN en terreno determinista vs estocástico.


In [ ]:
# ── Experimento: determinista vs estocástico ──
EPISODIOS_EXP = 1500    # converge ~1400; para el informe sube a 3000

def entrenar_resbalon(prob, episodios=EPISODIOS_EXP):
    e = TerrenoDRLEnv(RUTA_DEM, prob_resbalon=prob, semilla=SEMILLA)
    a = AgenteDQN(e.dim_obs, NUM_ACCIONES, device=device, semilla=SEMILLA)
    h = entrenar(e, a, episodios, eval_cada=100, verbose=False)
    return e, a, h

print("Entrenando DETERMINISTA (resbalón = 0.0) ...")
env_det, ag_det, h_det = entrenar_resbalon(0.0)
print("Entrenando ESTOCÁSTICO (resbalón = 0.25) ...")
env_sto, ag_sto, h_sto = entrenar_resbalon(0.25)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(h_det["eval_ep"], h_det["eval_exito"], "o-", label="determinista (resbalón 0.0)")
ax.plot(h_sto["eval_ep"], h_sto["eval_exito"], "s-", label="estocástico (resbalón 0.25)")
ax.set_xlabel("episodio"); ax.set_ylabel("éxito en examen"); ax.set_ylim(0, 1.05)
ax.set_title("DQN: terreno determinista vs estocástico"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

print(f"Final determinista: {h_det['eval_exito'][-1]:.0%} éxito, {h_det['eval_pasos'][-1]:.0f} pasos")
print(f"Final estocástico:  {h_sto['eval_exito'][-1]:.0%} éxito, {h_sto['eval_pasos'][-1]:.0f} pasos")
print("El estocástico suele llegar a algo menos de 100% y en más pasos: el azar cuesta,")
print("pero el agente aprende una política robusta pese al resbalón.")


## 7 · Reproducibilidad

- Semilla fija (42) en NumPy, PyTorch y el entorno; pares de examen con semilla propia (12345).
- DEM: Copernicus GLO-30, tile S17/W072, bbox (−71.475, −16.375, −71.445, −16.345), vía AWS Open Data.
- Dependencias: `numpy`, `matplotlib`, `rasterio`, `torch`.
- Pesos entrenados: `dqn_base.pt` (celda de abajo para descargarlos en Colab).

### Referencias
- Mnih, V. et al. *Human-level control through deep reinforcement learning*. Nature 518 (2015).
- Liu, R. & Zou, J. *The effects of memory replay in reinforcement learning*. Allerton (2018).
- Van Hasselt, H. et al. *Deep RL with double Q-learning*. AAAI (2016).
- Sutton, R. & Barto, A. *Reinforcement Learning: An Introduction* (2018).
- Slides MIA-204, clase 5.1 (MEng. María Fernanda Tejada Begazo).


In [ ]:
# ── (Colab) descargar los pesos entrenados ─────────────────────────────
try:
    from google.colab import files
    files.download("dqn_base.pt")
except ImportError:
    print("Local: los pesos ya están en ./dqn_base.pt")
